In [ ]:

import pandas as pd
from io import BytesIO
from google.colab import files

# ==================================================
# STEP 1: UPLOAD AND LOAD DATA
# ==================================================

uploaded = files.upload()
filename = next(iter(uploaded))

columns = [
    "HourUTC",
    "FullDate",
    "HourOfDay",
    "DemandMW",
    "TemperatureC",
    "HumidityPercent",
    "PrecipitationMM"
]

df = pd.read_csv(
    BytesIO(uploaded[filename]),
    header=None,
    names=columns
)

# ==================================================
# STEP 2: DATA TYPE CONVERSION
# ==================================================

df["HourUTC"] = pd.to_datetime(
    df["HourUTC"], errors="coerce"
)

df["FullDate"] = pd.to_datetime(
    df["FullDate"], errors="coerce"
)

numeric_cols = [
    "HourOfDay",
    "DemandMW",
    "TemperatureC",
    "HumidityPercent",
    "PrecipitationMM"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col], errors="coerce"
    )

# ==================================================
# STEP 3: SORT AND VALIDATE DATA
# ==================================================

df = df.sort_values("HourUTC").reset_index(drop=True)
df["Year"] = df["FullDate"].dt.year

print("=== DATASET SUMMARY ===")
print("File:", filename)
print("Total Rows:", len(df))
print("Total Columns (original):", len(columns))
print("Unique Hours:", df["HourUTC"].nunique())
print("Duplicate Hours:", df["HourUTC"].duplicated().sum())

# ==================================================
# STEP 4: MISSING VALUES
# ==================================================

print("\n=== MISSING VALUES ===")
print(df[columns].isnull().sum())

# ==================================================
# STEP 5: TIME CONTINUITY
# ==================================================

time_diff = df["HourUTC"].diff()
gaps = time_diff[time_diff > pd.Timedelta(hours=1)]

missing_hours = (
    gaps / pd.Timedelta(hours=1) - 1
).sum()

print("\n=== TIME CONTINUITY ===")
print("Number of gaps:", len(gaps))
print("Total missing hours:", int(missing_hours))

# ==================================================
# STEP 6: RECORDS BY YEAR
# ==================================================

print("\n=== RECORDS BY YEAR ===")
print(df.groupby("Year").size())

# ==================================================
# STEP 7: DESCRIPTIVE STATISTICS
# ==================================================

print("\n=== NUMERIC STATISTICS ===")
display(df[
    ["DemandMW", "TemperatureC",
     "HumidityPercent", "PrecipitationMM"]
].describe())

# ==================================================
# STEP 8: PREVIEW
# ==================================================

print("\n=== FIRST 5 ROWS ===")
display(df.head())


Saving Energy_ML_Hourly.csv to Energy_ML_Hourly.csv
=== DATASET SUMMARY ===
File: Energy_ML_Hourly.csv
Total Rows: 61306
Total Columns (original): 7
Unique Hours: 61306
Duplicate Hours: 0

=== MISSING VALUES ===
HourUTC            0
FullDate           0
HourOfDay          0
DemandMW           0
TemperatureC       0
HumidityPercent    0
PrecipitationMM    0
dtype: int64

=== TIME CONTINUITY ===
Number of gaps: 11
Total missing hours: 61

=== RECORDS BY YEAR ===
Year
2019    8712
2020    8782
2021    8758
2022    8757
2023    8757
2024    8782
2025    8758
dtype: int64

=== NUMERIC STATISTICS ===


,DemandMW,TemperatureC,HumidityPercent,PrecipitationMM
count,61306.000000,61306.000000,61306.000000,61306.000000
mean,21711.123831,7.574342,72.314325,0.134338
std,5267.977400,11.921354,16.623961,0.505923
min,13293.750000,-32.300000,12.000000,0.000000
25%,17836.440000,-1.100000,61.000000,0.000000
50%,19951.335000,8.100000,75.000000,0.000000
75%,25598.125000,17.400000,86.000000,0.000000
max,42472.830000,35.900000,100.000000,23.400000



=== FIRST 5 ROWS ===


,HourUTC,FullDate,HourOfDay,DemandMW,TemperatureC,HumidityPercent,PrecipitationMM,Year
0,2019-01-01 06:00:00,2019-01-01,1,23762.55,0.8,98.0,1.3,2019
1,2019-01-01 07:00:00,2019-01-01,2,23830.23,0.8,96.0,2.0,2019
2,2019-01-01 08:00:00,2019-01-01,3,23608.07,0.9,98.0,3.1,2019
3,2019-01-01 09:00:00,2019-01-01,4,23562.48,0.5,99.0,2.8,2019
4,2019-01-01 10:00:00,2019-01-01,5,23546.16,0.7,94.0,2.9,2019


In [ ]:

# ==============================================
# STEP 2: TIME GAP ANALYSIS
# ==============================================

df = df.sort_values("HourUTC").reset_index(drop=True)

# Previous timestamp
previous_hour = df["HourUTC"].shift(1)

# Difference between consecutive timestamps
hour_diff = (
    df["HourUTC"] - previous_hour
).dt.total_seconds() / 3600

# Find gaps larger than one hour
gap_mask = hour_diff > 1

# Create gap report
gap_report = pd.DataFrame({
    "PreviousHourUTC": previous_hour[gap_mask].values,
    "NextHourUTC": df.loc[gap_mask, "HourUTC"].values,
    "MissingHours": (
        hour_diff[gap_mask] - 1
    ).astype(int).values
})

# Sort by largest gaps
gap_report = gap_report.sort_values(
    "MissingHours",
    ascending=False
).reset_index(drop=True)

print("=== TIME GAP SUMMARY ===")
print("Number of gaps:", len(gap_report))
print("Total missing hours:",
      gap_report["MissingHours"].sum())

print("\n=== GAP DETAILS ===")
display(gap_report)


=== TIME GAP SUMMARY ===
Number of gaps: 11
Total missing hours: 61

=== GAP DETAILS ===


,PreviousHourUTC,NextHourUTC,MissingHours
0,2019-06-21 18:00:00,2019-06-23 12:00:00,41
1,2019-11-11 18:00:00,2019-11-11 23:00:00,4
2,2019-11-03 04:00:00,2019-11-03 07:00:00,2
3,2020-11-01 04:00:00,2020-11-01 07:00:00,2
4,2021-11-07 04:00:00,2021-11-07 07:00:00,2
5,2022-11-06 04:00:00,2022-11-06 07:00:00,2
6,2023-11-05 04:00:00,2023-11-05 07:00:00,2
7,2025-11-02 04:00:00,2025-11-02 07:00:00,2
8,2024-11-03 04:00:00,2024-11-03 07:00:00,2
9,2022-01-01 04:00:00,2022-01-01 06:00:00,1


In [ ]:

import numpy as np
import pandas as pd

# ============================================
# STEP 3: FEATURE ENGINEERING
# ============================================

# 1. Create a complete hourly UTC timeline
hourly = (
    df.sort_values("HourUTC")
      .set_index("HourUTC")
      .asfreq("h")
)

# Mark rows with missing demand
hourly["IsMissingHour"] = hourly["DemandMW"].isna()

# 2. Convert UTC timestamps to Quebec local time
local_time = (
    hourly.index.tz_localize("UTC")
                .tz_convert("America/Toronto")
)

# 3. Calendar features
hourly["Hour"] = local_time.hour
hourly["DayOfWeek"] = local_time.dayofweek
hourly["Month"] = local_time.month
hourly["Year"] = local_time.year

hourly["IsWeekend"] = (
    hourly["DayOfWeek"] >= 5
).astype(int)

# 4. Lag features
# Exact previous hours on the UTC timeline
hourly["Lag_1"] = hourly["DemandMW"].shift(1)
hourly["Lag_24"] = hourly["DemandMW"].shift(24)
hourly["Lag_168"] = hourly["DemandMW"].shift(168)

# 5. Rolling features
# shift(1) prevents using current demand
hourly["RollingMean_24"] = (
    hourly["DemandMW"]
    .shift(1)
    .rolling(window=24, min_periods=24)
    .mean()
)

hourly["RollingMean_168"] = (
    hourly["DemandMW"]
    .shift(1)
    .rolling(window=168, min_periods=168)
    .mean()
)

# 6. Prepare ML dataset
# Keep only observed target values and valid lags
feature_cols = [
    "Hour",
    "DayOfWeek",
    "Month",
    "IsWeekend",
    "Lag_1",
    "Lag_24",
    "Lag_168",
    "RollingMean_24",
    "RollingMean_168"
]

ml_df = hourly.dropna(
    subset=["DemandMW"] + feature_cols
).copy()

ml_df = ml_df.reset_index()

# 7. Validation summary
print("=== FEATURE ENGINEERING SUMMARY ===")

print("Original rows:", len(df))
print("Complete hourly timeline:", len(hourly))
print("Missing hourly slots:",
      int(hourly["IsMissingHour"].sum()))
print("ML-ready rows:", len(ml_df))

print("\n=== ML RECORDS BY YEAR ===")
print(ml_df.groupby("Year").size())

print("\n=== MISSING FEATURE VALUES ===")
print(ml_df[feature_cols].isna().sum())

print("\n=== FIRST 5 ROWS ===")
display(
    ml_df[
        ["HourUTC", "DemandMW"] + feature_cols
    ].head()
)


=== FEATURE ENGINEERING SUMMARY ===
Original rows: 61306
Complete hourly timeline: 61367
Missing hourly slots: 61
ML-ready rows: 59290

=== ML RECORDS BY YEAR ===
Year
2019    8040
2020    8614
2021    8590
2022    8421
2023    8421
2024    8614
2025    8590
dtype: int64

=== MISSING FEATURE VALUES ===
Hour               0
DayOfWeek          0
Month              0
IsWeekend          0
Lag_1              0
Lag_24             0
Lag_168            0
RollingMean_24     0
RollingMean_168    0
dtype: int64

=== FIRST 5 ROWS ===


,HourUTC,DemandMW,Hour,DayOfWeek,Month,IsWeekend,Lag_1,Lag_24,Lag_168,RollingMean_24,RollingMean_168
0,2019-01-08 06:00:00,28190.58,1,1,1,0,28674.67,26658.05,23762.55,30640.476250,27093.126667
1,2019-01-08 07:00:00,27998.60,2,1,1,0,28190.58,26848.81,23830.23,30704.331667,27119.483988
2,2019-01-08 08:00:00,27975.88,3,1,1,0,27998.60,27314.97,23608.07,30752.239583,27144.295714
3,2019-01-08 09:00:00,27811.19,4,1,1,0,27975.88,27836.78,23562.48,30779.777500,27170.294583
4,2019-01-08 10:00:00,27845.25,5,1,1,0,27811.19,28491.30,23546.16,30778.711250,27195.584524


In [ ]:

# ============================================
# STEP 4: TRAIN / VALIDATION / TEST SPLIT
# ============================================

# 1. Split data chronologically
train_df = ml_df[
    ml_df["Year"].between(2019, 2023)
].copy()

val_df = ml_df[
    ml_df["Year"] == 2024
].copy()

test_df = ml_df[
    ml_df["Year"] == 2025
].copy()

# 2. Prepare input features (X) and target (y)
target = "DemandMW"

X_train = train_df[feature_cols]
y_train = train_df[target]

X_val = val_df[feature_cols]
y_val = val_df[target]

X_test = test_df[feature_cols]
y_test = test_df[target]

# 3. Validate chronological order
assert train_df["HourUTC"].max() < val_df["HourUTC"].min()
assert val_df["HourUTC"].max() < test_df["HourUTC"].min()

# 4. Display results
print("=== DATA SPLIT SUMMARY ===")

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Testing rows:", len(X_test))

print("\n=== DATE RANGES ===")

for name, data in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]:
    print(
        f"{name}: "
        f"{data['HourUTC'].min()} "
        f"to {data['HourUTC'].max()}"
    )

print("\n=== MODEL FEATURES ===")
print(feature_cols)

print("\n=== TARGET ===")
print(target)


=== DATA SPLIT SUMMARY ===
Training rows: 42086
Validation rows: 8614
Testing rows: 8590

=== DATE RANGES ===
Training: 2019-01-08 06:00:00 to 2024-01-01 04:00:00
Validation: 2024-01-01 05:00:00 to 2025-01-01 04:00:00
Testing: 2025-01-01 05:00:00 to 2026-01-01 04:00:00

=== MODEL FEATURES ===
['Hour', 'DayOfWeek', 'Month', 'IsWeekend', 'Lag_1', 'Lag_24', 'Lag_168', 'RollingMean_24', 'RollingMean_168']

=== TARGET ===
DemandMW


In [ ]:

# ============================================
# STEP 5: NAIVE BASELINE MODEL
# ============================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Actual demand in validation period (2024)
y_actual = y_val

# 2. Baseline prediction:
# Demand at the same UTC hour 24 hours earlier
y_pred_baseline = val_df["Lag_24"]

# 3. Calculate evaluation metrics
mae = mean_absolute_error(
    y_actual, y_pred_baseline
)

rmse = np.sqrt(
    mean_squared_error(y_actual, y_pred_baseline)
)

r2 = r2_score(
    y_actual, y_pred_baseline
)

# 4. Store results for model comparison
baseline_results = pd.DataFrame([{
    "Model": "Naive Baseline (Lag 24)",
    "MAE": mae,
    "RMSE": rmse,
    "R2": r2
}])

print("=== BASELINE MODEL RESULTS ===")
print("Validation Year: 2024")
print(f"MAE:  {mae:,.2f} MW")
print(f"RMSE: {rmse:,.2f} MW")
print(f"R2:   {r2:.4f}")

display(baseline_results)

# 5. Preview actual vs predicted values
baseline_predictions = pd.DataFrame({
    "HourUTC": val_df["HourUTC"].values,
    "ActualDemandMW": y_actual.values,
    "PredictedDemandMW": y_pred_baseline.values
})

print("\n=== ACTUAL VS PREDICTED ===")
display(baseline_predictions.head(10))


=== BASELINE MODEL RESULTS ===
Validation Year: 2024
MAE:  1,027.94 MW
RMSE: 1,501.23 MW
R2:   0.9056


,Model,MAE,RMSE,R2
0,Naive Baseline (Lag 24),1027.937287,1501.225705,0.905625



=== ACTUAL VS PREDICTED ===


,HourUTC,ActualDemandMW,PredictedDemandMW
0,2024-01-01 05:00:00,25484.99,24435.64
1,2024-01-01 06:00:00,25342.26,24100.67
2,2024-01-01 07:00:00,25324.98,24045.82
3,2024-01-01 08:00:00,25247.73,24103.74
4,2024-01-01 09:00:00,25310.01,24215.88
5,2024-01-01 10:00:00,25497.10,24596.15
6,2024-01-01 11:00:00,26072.13,25075.03
7,2024-01-01 12:00:00,26694.50,25887.53
8,2024-01-01 13:00:00,27135.59,26587.13
9,2024-01-01 14:00:00,27209.57,27191.56


In [ ]:

# =================================================
# STEP 6: RANDOM FOREST REGRESSOR
# =================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Initialize model
rf_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# 2. Train on 2019-2023 data
rf_model.fit(X_train, y_train)

# 3. Predict validation year (2024)
y_pred_rf = rf_model.predict(X_val)

# 4. Evaluate model
rf_mae = mean_absolute_error(y_val, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_val, y_pred_rf))
rf_r2 = r2_score(y_val, y_pred_rf)

# 5. Store results
rf_results = pd.DataFrame([{
    "Model": "Random Forest",
    "MAE": rf_mae,
    "RMSE": rf_rmse,
    "R2": rf_r2
}])

# 6. Compare models
model_comparison = pd.concat(
    [baseline_results, rf_results],
    ignore_index=True
)

print("=== RANDOM FOREST RESULTS ===")
print("Validation Year: 2024")
print(f"MAE:  {rf_mae:,.2f} MW")
print(f"RMSE: {rf_rmse:,.2f} MW")
print(f"R2:   {rf_r2:.4f}")

print("\n=== MODEL COMPARISON ===")
display(model_comparison.round(4))

# 7. Feature importance
feature_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

print("\n=== FEATURE IMPORTANCE ===")
display(feature_importance)

# 8. Preview predictions
rf_predictions = pd.DataFrame({
    "HourUTC": val_df["HourUTC"].values,
    "ActualDemandMW": y_val.values,
    "PredictedDemandMW": y_pred_rf
})

print("\n=== ACTUAL VS PREDICTED ===")
display(rf_predictions.head(10))


=== RANDOM FOREST RESULTS ===
Validation Year: 2024
MAE:  175.85 MW
RMSE: 251.54 MW
R2:   0.9974

=== MODEL COMPARISON ===


,Model,MAE,RMSE,R2
0,Naive Baseline (Lag 24),1027.9373,1501.2257,0.9056
1,Random Forest,175.8453,251.5364,0.9974



=== FEATURE IMPORTANCE ===


,Feature,Importance
4,Lag_1,0.984356
0,Hour,0.012140
5,Lag_24,0.001530
7,RollingMean_24,0.000508
2,Month,0.000383
6,Lag_168,0.000340
8,RollingMean_168,0.000308
1,DayOfWeek,0.000303
3,IsWeekend,0.000132



=== ACTUAL VS PREDICTED ===


,HourUTC,ActualDemandMW,PredictedDemandMW
0,2024-01-01 05:00:00,25484.99,25044.570788
1,2024-01-01 06:00:00,25342.26,25073.815633
2,2024-01-01 07:00:00,25324.98,25274.578204
3,2024-01-01 08:00:00,25247.73,25476.725477
4,2024-01-01 09:00:00,25310.01,25507.751980
5,2024-01-01 10:00:00,25497.10,25851.123401
6,2024-01-01 11:00:00,26072.13,26649.131063
7,2024-01-01 12:00:00,26694.50,27862.725556
8,2024-01-01 13:00:00,27135.59,27913.192541
9,2024-01-01 14:00:00,27209.57,26870.966363


In [ ]:

# ==================================================
# STEP 7: LAG-1 BASELINE & GRADIENT BOOSTING
# ==================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Evaluation function
def evaluate_model(name, actual, predicted):
    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "R2": r2_score(actual, predicted)
    }

# 2. Naive Baseline: previous hour
y_pred_lag1 = val_df["Lag_1"].values

lag1_results = pd.DataFrame([
    evaluate_model(
        "Naive Baseline (Lag 1)",
        y_val,
        y_pred_lag1
    )
])

# 3. Initialize Gradient Boosting model
gb_model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.06,
    max_leaf_nodes=31,
    min_samples_leaf=25,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

# 4. Train on 2019-2023
gb_model.fit(X_train, y_train)

# 5. Predict validation year (2024)
y_pred_gb = gb_model.predict(X_val)

# 6. Evaluate Gradient Boosting
gb_results = pd.DataFrame([
    evaluate_model(
        "Gradient Boosting",
        y_val,
        y_pred_gb
    )
])

# 7. Compare all models
model_comparison = pd.concat(
    [
        baseline_results,
        lag1_results,
        rf_results,
        gb_results
    ],
    ignore_index=True
)

model_comparison = model_comparison.sort_values(
    "RMSE"
).reset_index(drop=True)

print("=== MODEL COMPARISON: VALIDATION 2024 ===")
display(model_comparison.round(4))

# 8. Best model on validation data
best_model_name = model_comparison.iloc[0]["Model"]

print("\nBest validation model:", best_model_name)

# 9. Gradient Boosting predictions preview
gb_predictions = pd.DataFrame({
    "HourUTC": val_df["HourUTC"].values,
    "ActualDemandMW": y_val.values,
    "PredictedDemandMW": y_pred_gb
})

print("\n=== GRADIENT BOOSTING PREDICTIONS ===")
display(gb_predictions.head(10))


=== MODEL COMPARISON: VALIDATION 2024 ===


,Model,MAE,RMSE,R2
0,Gradient Boosting,164.8210,230.0202,0.9978
1,Random Forest,175.8453,251.5364,0.9974
2,Naive Baseline (Lag 1),497.1123,644.2193,0.9826
3,Naive Baseline (Lag 24),1027.9373,1501.2257,0.9056



Best validation model: Gradient Boosting

=== GRADIENT BOOSTING PREDICTIONS ===


,HourUTC,ActualDemandMW,PredictedDemandMW
0,2024-01-01 05:00:00,25484.99,25137.914998
1,2024-01-01 06:00:00,25342.26,25084.408453
2,2024-01-01 07:00:00,25324.98,25329.171485
3,2024-01-01 08:00:00,25247.73,25464.616167
4,2024-01-01 09:00:00,25310.01,25555.082619
5,2024-01-01 10:00:00,25497.10,25820.139668
6,2024-01-01 11:00:00,26072.13,26509.658319
7,2024-01-01 12:00:00,26694.50,27922.662643
8,2024-01-01 13:00:00,27135.59,27841.648796
9,2024-01-01 14:00:00,27209.57,27130.671488


In [ ]:

# ==================================================
# STEP 8: FINAL TRAINING & TESTING (2025)
# ==================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 1. Combine Training + Validation (2019-2024)
final_train_df = pd.concat(
    [train_df, val_df],
    ignore_index=True
).sort_values("HourUTC")

X_final_train = final_train_df[feature_cols]
y_final_train = final_train_df["DemandMW"]

# 2. Keep 2025 only for final testing
X_final_test = test_df[feature_cols]
y_final_test = test_df["DemandMW"]

assert final_train_df["HourUTC"].max() < test_df["HourUTC"].min()
assert len(X_final_train) == 50700
assert len(X_final_test) == 8590

# 3. Create final Gradient Boosting model
# Same hyperparameters selected during validation
final_model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.06,
    max_leaf_nodes=31,
    min_samples_leaf=25,
    l2_regularization=1.0,
    early_stopping=False,
    random_state=42
)

# 4. Train on 2019-2024
final_model.fit(X_final_train, y_final_train)

# 5. Predict 2025
y_pred_final = final_model.predict(X_final_test)

# 6. Evaluation function
def final_evaluate(name, actual, predicted):
    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "R2": r2_score(actual, predicted)
    }

# 7. Evaluate all three methods on 2025
final_results = pd.DataFrame([
    final_evaluate(
        "Gradient Boosting",
        y_final_test,
        y_pred_final
    ),
    final_evaluate(
        "Naive Baseline (Lag 1)",
        y_final_test,
        test_df["Lag_1"]
    ),
    final_evaluate(
        "Naive Baseline (Lag 24)",
        y_final_test,
        test_df["Lag_24"]
    )
]).sort_values("RMSE").reset_index(drop=True)

print("=== FINAL TEST RESULTS: 2025 ===")
display(final_results.round(4))

# 8. Save predictions in memory for Power BI
final_predictions = pd.DataFrame({
    "HourUTC": test_df["HourUTC"].values,
    "ActualDemandMW": y_final_test.values,
    "PredictedDemandMW": y_pred_final
})

final_predictions["AbsoluteErrorMW"] = abs(
    final_predictions["ActualDemandMW"]
    - final_predictions["PredictedDemandMW"]
)

print("\n=== ACTUAL VS PREDICTED ===")
display(final_predictions.head(10))

print("\nFinal training rows:", len(X_final_train))
print("Final testing rows:", len(X_final_test))


=== FINAL TEST RESULTS: 2025 ===


,Model,MAE,RMSE,R2
0,Gradient Boosting,188.5147,284.0097,0.9977
1,Naive Baseline (Lag 1),504.8835,654.0518,0.9879
2,Naive Baseline (Lag 24),1149.7201,1633.0632,0.9247



=== ACTUAL VS PREDICTED ===


,HourUTC,ActualDemandMW,PredictedDemandMW,AbsoluteErrorMW
0,2025-01-01 05:00:00,21598.78,21067.839325,530.940675
1,2025-01-01 06:00:00,21445.77,21198.903865,246.866135
2,2025-01-01 07:00:00,21486.31,21303.104944,183.205056
3,2025-01-01 08:00:00,21444.76,21462.973982,18.213982
4,2025-01-01 09:00:00,21438.65,21611.082330,172.432330
5,2025-01-01 10:00:00,21594.44,21804.747777,210.307777
6,2025-01-01 11:00:00,21961.99,22462.208567,500.218567
7,2025-01-01 12:00:00,22516.90,23559.267803,1042.367803
8,2025-01-01 13:00:00,22810.03,23602.701731,792.671731
9,2025-01-01 14:00:00,23161.19,22998.661720,162.528280



Final training rows: 50700
Final testing rows: 8590


In [ ]:

# ==================================================
# STEP 9: EXPORT ML RESULTS FOR POWER BI
# ==================================================

import pandas as pd
from google.colab import files

# 1. Prepare prediction results
export_df = final_predictions.copy()

# 2. Convert UTC to Quebec local time
export_df["HourUTC"] = pd.to_datetime(
    export_df["HourUTC"]
)

local_time = (
    export_df["HourUTC"]
    .dt.tz_localize("UTC")
    .dt.tz_convert("America/Toronto")
)

# 3. Add calendar columns for Power BI
export_df["FullDate"] = local_time.dt.strftime(
    "%Y-%m-%d"
)

export_df["CalendarYear"] = local_time.dt.year
export_df["CalendarMonth"] = local_time.dt.month
export_df["MonthShort"] = local_time.dt.strftime("%b")
export_df["LocalHour"] = local_time.dt.hour

# 4. Add prediction details
export_df["Model"] = "Gradient Boosting"

export_df["ErrorMW"] = (
    export_df["ActualDemandMW"]
    - export_df["PredictedDemandMW"]
)

export_df["AbsoluteErrorMW"] = export_df["ErrorMW"].abs()

# 5. Select final columns
export_df = export_df[
    [
        "HourUTC",
        "FullDate",
        "CalendarYear",
        "CalendarMonth",
        "MonthShort",
        "LocalHour",
        "Model",
        "ActualDemandMW",
        "PredictedDemandMW",
        "ErrorMW",
        "AbsoluteErrorMW"
    ]
]

# 6. Prepare model comparison results
comparison_export = final_results.copy()
comparison_export["TestYear"] = 2025

# 7. Validate before exporting
assert len(export_df) == 8590
assert export_df["HourUTC"].is_unique
assert export_df.isna().sum().sum() == 0

# 8. Save CSV files
export_df.to_csv(
    "Energy_Forecast_2025.csv",
    index=False
)

comparison_export.to_csv(
    "Model_Comparison_2025.csv",
    index=False
)

# 9. Preview
print("=== EXPORT SUMMARY ===")
print("Forecast rows:", len(export_df))
print("Comparison rows:", len(comparison_export))

print("\n=== FORECAST PREVIEW ===")
display(export_df.head())

print("\n=== MODEL COMPARISON ===")
display(comparison_export)

# 10. Download files
files.download("Energy_Forecast_2025.csv")
files.download("Model_Comparison_2025.csv")


=== EXPORT SUMMARY ===
Forecast rows: 8590
Comparison rows: 3

=== FORECAST PREVIEW ===


,HourUTC,FullDate,CalendarYear,CalendarMonth,MonthShort,LocalHour,Model,ActualDemandMW,PredictedDemandMW,ErrorMW,AbsoluteErrorMW
0,2025-01-01 05:00:00,2025-01-01,2025,1,Jan,0,Gradient Boosting,21598.78,21067.839325,530.940675,530.940675
1,2025-01-01 06:00:00,2025-01-01,2025,1,Jan,1,Gradient Boosting,21445.77,21198.903865,246.866135,246.866135
2,2025-01-01 07:00:00,2025-01-01,2025,1,Jan,2,Gradient Boosting,21486.31,21303.104944,183.205056,183.205056
3,2025-01-01 08:00:00,2025-01-01,2025,1,Jan,3,Gradient Boosting,21444.76,21462.973982,-18.213982,18.213982
4,2025-01-01 09:00:00,2025-01-01,2025,1,Jan,4,Gradient Boosting,21438.65,21611.082330,-172.432330,172.432330



=== MODEL COMPARISON ===


,Model,MAE,RMSE,R2,TestYear
0,Gradient Boosting,188.514723,284.009687,0.997722,2025
1,Naive Baseline (Lag 1),504.883545,654.051829,0.987920,2025
2,Naive Baseline (Lag 24),1149.720107,1633.063246,0.924693,2025


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# =====================================================
# STEP 10: SEVEN-DAY ELECTRICITY DEMAND FORECAST
# Quebec Energy Intelligence
# =====================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from google.colab import files

# -----------------------------------------------------
# 1. Prepare a complete hourly UTC timeline
# -----------------------------------------------------

data = df[["HourUTC", "DemandMW"]].copy()

data["HourUTC"] = pd.to_datetime(
    data["HourUTC"], utc=True
)

hourly = (
    data.sort_values("HourUTC")
        .drop_duplicates("HourUTC")
        .set_index("HourUTC")
        .asfreq("h")
)

# -----------------------------------------------------
# 2. Feature engineering
# Use only information available before forecast time
# -----------------------------------------------------

def build_features(frame):
    result = frame.copy()
    local = result.index.tz_convert("America/Toronto")

    result["Year"] = local.year
    result["Hour"] = local.hour
    result["DayOfWeek"] = local.dayofweek
    result["Month"] = local.month
    result["IsWeekend"] = (
        local.dayofweek >= 5
    ).astype(int)

    result["Lag_168"] = result["DemandMW"].shift(168)
    result["Lag_336"] = result["DemandMW"].shift(336)
    result["Lag_504"] = result["DemandMW"].shift(504)

    return result

feature_cols_7d = [
    "Hour",
    "DayOfWeek",
    "Month",
    "IsWeekend",
    "Lag_168",
    "Lag_336",
    "Lag_504"
]

features = build_features(hourly)

ready = features.dropna(
    subset=["DemandMW"] + feature_cols_7d
).copy()

# -----------------------------------------------------
# 3. Chronological backtest on 2025
# Train: 2019-2024
# Test: 2025
# -----------------------------------------------------

train = ready[ready["Year"] <= 2024]
test = ready[ready["Year"] == 2025]

assert not train.empty and not test.empty
assert train.index.max() < test.index.min()

def make_model():
    return HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.06,
        max_leaf_nodes=31,
        min_samples_leaf=25,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=42
    )

backtest_model = make_model()

backtest_model.fit(
    train[feature_cols_7d],
    train["DemandMW"]
)

y_actual = test["DemandMW"]
y_pred = backtest_model.predict(
    test[feature_cols_7d]
)

# Weekly naive baseline for comparison
y_weekly = test["Lag_168"]

def metrics(name, actual, predicted):
    return {
        "Model": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "R2": r2_score(actual, predicted)
    }

results_7d = pd.DataFrame([
    metrics("Weekly Gradient Boosting", y_actual, y_pred),
    metrics("Naive Weekly Baseline", y_actual, y_weekly)
])

print("=== SEVEN-DAY MODEL BACKTEST: 2025 ===")
display(results_7d.round(4))

# -----------------------------------------------------
# 4. Retrain on all observed historical data
# -----------------------------------------------------

final_7d_model = make_model()

final_7d_model.fit(
    ready[feature_cols_7d],
    ready["DemandMW"]
)

# -----------------------------------------------------
# 5. Create next 168 hourly timestamps
# -----------------------------------------------------

cutoff = hourly["DemandMW"].last_valid_index()

future_hours = pd.date_range(
    start=cutoff + pd.Timedelta(hours=1),
    periods=168,
    freq="h",
    tz="UTC"
)

extended = hourly.reindex(
    hourly.index.union(future_hours)
)

future_features = build_features(extended).loc[
    future_hours, feature_cols_7d
]

if future_features.isna().any().any():
    raise ValueError(
        "Missing lag features in forecast horizon."
    )

# -----------------------------------------------------
# 6. Predict seven days without future actual demand
# -----------------------------------------------------

future_predictions = final_7d_model.predict(
    future_features
)

forecast_7d = pd.DataFrame({
    "HourUTC": future_hours.tz_localize(None),
    "ForecastOriginUTC": cutoff.tz_localize(None),
    "HorizonHours": np.arange(1, 169),
    "PredictedDemandMW": future_predictions,
    "Model": "Weekly Gradient Boosting"
})

forecast_7d["LocalDateTime"] = (
    future_hours
    .tz_convert("America/Toronto")
    .strftime("%Y-%m-%d %H:%M:%S")
)

# -----------------------------------------------------
# 7. Validation and export
# -----------------------------------------------------

assert len(forecast_7d) == 168
assert forecast_7d["HourUTC"].is_unique
assert forecast_7d["PredictedDemandMW"].notna().all()

print("\n=== FUTURE FORECAST SUMMARY ===")
print("Last observed UTC:", cutoff)
print("Forecast hours:", len(forecast_7d))
print("Start UTC:", forecast_7d["HourUTC"].min())
print("End UTC:", forecast_7d["HourUTC"].max())

print("\n=== PREDICTION PREVIEW ===")
display(forecast_7d.head(10))

forecast_7d.to_csv(
    "Forecast_Next_7_Days.csv",
    index=False
)

files.download("Forecast_Next_7_Days.csv")


=== SEVEN-DAY MODEL BACKTEST: 2025 ===


,Model,MAE,RMSE,R2
0,Weekly Gradient Boosting,1572.3931,2363.6383,0.8396
1,Naive Weekly Baseline,1844.9892,2716.4807,0.7881



=== FUTURE FORECAST SUMMARY ===
Last observed UTC: 2026-01-01 04:00:00+00:00
Forecast hours: 168
Start UTC: 2026-01-01 05:00:00
End UTC: 2026-01-08 04:00:00

=== PREDICTION PREVIEW ===


,HourUTC,ForecastOriginUTC,HorizonHours,PredictedDemandMW,Model,LocalDateTime
0,2026-01-01 05:00:00,2026-01-01 04:00:00,1,27100.403347,Weekly Gradient Boosting,2026-01-01 00:00:00
1,2026-01-01 06:00:00,2026-01-01 04:00:00,2,27042.705384,Weekly Gradient Boosting,2026-01-01 01:00:00
2,2026-01-01 07:00:00,2026-01-01 04:00:00,3,27244.145811,Weekly Gradient Boosting,2026-01-01 02:00:00
3,2026-01-01 08:00:00,2026-01-01 04:00:00,4,27244.145811,Weekly Gradient Boosting,2026-01-01 03:00:00
4,2026-01-01 09:00:00,2026-01-01 04:00:00,5,27163.993676,Weekly Gradient Boosting,2026-01-01 04:00:00
5,2026-01-01 10:00:00,2026-01-01 04:00:00,6,27416.901557,Weekly Gradient Boosting,2026-01-01 05:00:00
6,2026-01-01 11:00:00,2026-01-01 04:00:00,7,28997.202207,Weekly Gradient Boosting,2026-01-01 06:00:00
7,2026-01-01 12:00:00,2026-01-01 04:00:00,8,30509.087944,Weekly Gradient Boosting,2026-01-01 07:00:00
8,2026-01-01 13:00:00,2026-01-01 04:00:00,9,32016.887961,Weekly Gradient Boosting,2026-01-01 08:00:00
9,2026-01-01 14:00:00,2026-01-01 04:00:00,10,31560.197316,Weekly Gradient Boosting,2026-01-01 09:00:00


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>